# 05 — Out-of-Sample Validation
Development 发现候选，Validation 选择有限参数，Test 只运行一次。查看 test 后不得修改阈值并继续称其为 test。

In [ ]:
from pathlib import Path
import sys
root=Path.cwd().resolve()
while root!=root.parent and not (root/'pyproject.toml').exists(): root=root.parent
sys.path.insert(0,str(root))
import matplotlib.pyplot as plt
from research.spxw_put_skew.analysis import load_study_context,frozen_parameter_validation,robustness_sensitivity
dataset,config,collection,panel,readiness,conclusion=load_study_context(root)
grid,frozen=frozen_parameter_validation(panel)
sensitivity=robustness_sensitivity(panel,float(config.commission_per_contract))
print('Ready:',readiness.ready,'reasons=',readiness.reasons)
display(grid.head(20))
display(frozen)
display(sensitivity)

In [ ]:
if len(grid):
 view=grid[(grid['sample']=='validation') & (~grid.vol_filter) & (~grid.trend_filter)]
 if len(view):
  ax=view.plot(x='threshold',y='mean_pnl',marker='o',title='Validation parameter neighborhood')
  ax.grid(alpha=.2); plt.tight_layout()

In [ ]:
print('Hypothesis status:',conclusion.status)
print(conclusion.message)
if not readiness.ready: print('TEST CONCLUSION LOCKED: real-data readiness has not passed.')

最终接受条件还包括 block-bootstrap 95% CI 下界大于 0、高 skew 后 skew 均值回复，以及 test 期条件策略优于无条件基准。稳健性必须覆盖双倍费用、额外 0.05 点滑点、延迟 2 slices 和组合压力；参数必须呈稳定邻域，不能只依赖单点最优。